In [6]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import json
print(f"GPU available: {torch.cuda.is_available()}")


GPU available: True


In [7]:
# read csv
df = pd.read_csv("../dataset/vibe_coding_search.csv")
# drop empty text and duplicate text to preserve comments that dh ID
df = df.dropna(subset=['Text'])
df = df.drop_duplicates(subset=['Text'])

df['Clean_Text'] = df['Text'].astype(str).str.lower()

print(f"Total number of unique records {len(df)}")

Total number of unique records 75187


In [8]:
# Lock in the Parent_IDs while the original CSV sequence is intact
post_ids = df['ID'].where(df['Type'].str.lower() == 'post', np.nan)
df['Parent_ID'] = np.where(df['Type'].str.lower() == 'post', None, post_ids.ffill())

# Generate missing comment IDs safely
missing_ids = df['ID'].isna()
if missing_ids.any():
    df.loc[missing_ids, 'ID'] = [f"comment_{i}" for i in range(missing_ids.sum())]

# Run the Keyword Blacklist
job_keywords = [
    'hiring', 'hire', 'apply now', 'salary', 'years of experience', 
    'job requirement', 'qualifications', 'full-time', 'benefits', 
    'recruiting', 'competitive pay', 'join our team', 'AI Accelerator',
    'HOWTO', 'howto', 'iptv', 'IPTV'
]

pattern = '|'.join(job_keywords)
df = df[~df['Clean_Text'].str.contains(pattern, case=False, na=False)]
print(f"Records remaining after Keyword Blacklist: {len(df)}")

# apply new schema and parentID logic
df['Source'], df['Title'] = 'Reddit', None
df['Word_Count'] = df['Clean_Text'].astype(str).apply(lambda x: len(x.split()))

target_schema = ['ID', 'Parent_ID', 'Source', 'Type', 'Author', 'Text', 'Score', 'Date', 'Clean_Text', 'Word_Count', 'Title']
df = df[target_schema]

print(f"Total unique records after applying schema: {len(df)}")

Records remaining after Keyword Blacklist: 71819
Total unique records after applying schema: 71819


In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Ideal sentence 
target_query = (
    "vibe coding and natural language programming, giving in to the vibes with Cursor Composer or Windsurf Cascade, "
    "accepting AI code without reading diffs, rapid prototyping for non-technical founders, "
    "the death of junior developer roles, imposter syndrome and AI skill rot, "
    "spaghetti code and technical debt from LLMs, revolutionary productivity vs security risks and vulnerabilities, "
    "autonomous coding agents like Cline and Roo Code, coding beyond human comprehension"
)

# Encode the target query into a mathematical vector space
target_embedding = model.encode(target_query, convert_to_tensor=True)

print("Model and target embedding loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 978.77it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and target embedding loaded!


In [10]:
# Isolate Posts and Comments
posts_df = df[df['Type'].str.lower() == 'post'].copy()
comments_df = df[df['Type'].str.lower() == 'comment'].copy()
print(f"length of posts: {len(posts_df)}")
print(f"length of comments: {len(comments_df)}")

# Processing Posts
post_embeddings = model.encode(posts_df['Clean_Text'].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True)
posts_df['Relevance_Score'] = util.cos_sim(target_embedding, post_embeddings)[0].cpu().numpy()

# Processing Comments
comment_embeddings = model.encode(comments_df['Clean_Text'].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True)
comments_df['Relevance_Score'] = util.cos_sim(target_embedding, comment_embeddings)[0].cpu().numpy()

print("\nTop 3 Most Relevant Posts:")
print(posts_df.nlargest(3, 'Relevance_Score')[['Text', 'Relevance_Score']])

print("\nTop 3 Most Relevant Comments:")
print(comments_df.nlargest(3, 'Relevance_Score')[['Text', 'Relevance_Score']])

length of posts: 25033
length of comments: 46786


Batches: 100%|██████████| 732/732 [00:11<00:00, 61.51it/s] 



Top 3 Most Relevant Posts:
                                                    Text  Relevance_Score
55626  Vibe coding is thriving. The tools powering it...         0.738632
32229  Vibe Coding" is just a trendy rebrand for ship...         0.710984
19228  I am a dev an non vibe coder dev does guys lik...         0.705783

Top 3 Most Relevant Comments:
                                                    Text  Relevance_Score
46911  Vibe coding is real coding. It is NOT „softwar...         0.746416
71415  A vibe coder if and when AI goes away will be ...         0.707690
46905  Are we really talking about non-developer vibe...         0.692666


In [11]:
def get_total_words(dataframe):
    return dataframe['Clean_Text'].apply(lambda x: len(str(x).split())).sum()

threshold = 0.40
step = 0.02
success = False

while threshold >= 0.10:
    # Filter posts based on the threshold
    valid_posts = posts_df[posts_df['Relevance_Score'] >= threshold]
    valid_post_ids = set(valid_posts['ID'])
    
    # Filter comments based on the threshold AND ensure their parent survived
    valid_comments = comments_df[
        (comments_df['Relevance_Score'] >= threshold) & 
        (comments_df['Parent_ID'].isin(valid_post_ids))
    ]
    
    # Combine back together just to check total constraints
    df_filtered = pd.concat([valid_posts, valid_comments])
    
    total_records = len(df_filtered)
    total_words = get_total_words(df_filtered)
    
    if total_records >= 40000 and total_words >= 100000:
        print(f"Success! Extracted threshold: {threshold:.2f}")
        print(f"Final Records: {total_records} (Posts: {len(valid_posts)}, Comments: {len(valid_comments)})")
        print(f"Final Words: {total_words}")
        success = True
        break
    else:
        print(f"Threshold {threshold:.2f} yielded {total_records} records. Lowering...")
        threshold -= step

if not success:
    print(f"\nWARNING: Target of 40k records not met.")
    print(f"Saving the best available at 0.10 threshold.")

Threshold 0.40 yielded 3136 records. Lowering...
Threshold 0.38 yielded 4230 records. Lowering...
Threshold 0.36 yielded 5399 records. Lowering...
Threshold 0.34 yielded 6829 records. Lowering...
Threshold 0.32 yielded 8478 records. Lowering...
Threshold 0.30 yielded 10250 records. Lowering...
Threshold 0.28 yielded 12180 records. Lowering...
Threshold 0.26 yielded 14344 records. Lowering...
Threshold 0.24 yielded 16630 records. Lowering...
Threshold 0.22 yielded 19223 records. Lowering...
Threshold 0.20 yielded 21936 records. Lowering...
Threshold 0.18 yielded 25040 records. Lowering...
Threshold 0.16 yielded 28404 records. Lowering...
Threshold 0.14 yielded 32124 records. Lowering...
Threshold 0.12 yielded 36227 records. Lowering...

Saving the best available at 0.10 threshold.


In [12]:
json_output = []

for _, post in valid_posts.iterrows():
    # Map data to schema
    post_dict = {
        "ID": str(post['ID']),
        "Source": str(post['Source']),
        "Type": str(post['Type']),
        "Author": str(post['Author']),
        "Title": post['Title'] if pd.notnull(post['Title']) else None,
        "Text": str(post['Text']),
        "Score": int(post['Score']) if pd.notnull(post['Score']) else 0,
        "Date": str(post['Date']),
        "Word_Count": int(post['Word_Count']),
        "Comments": []
    }
    
    # Filter comments belonging to this post
    post_comments = valid_comments[valid_comments['Parent_ID'] == post['ID']]
    
    # Map Comment data to schema
    comments_list = []
    for _, comment in post_comments.iterrows():
        comment_dict = {
            "comment_id": str(comment['ID']),
            "parent_id": str(comment['Parent_ID']),
            "Source": str(comment['Source']),
            "Author": str(comment['Author']),
            "Text": str(comment['Text']),
            "Score": int(comment['Score']) if pd.notnull(comment['Score']) else 0,
            "Date": str(comment['Date']),
            "Word_Count": int(comment['Word_Count'])
        }
        comments_list.append(comment_dict)
        
    post_dict['Comments'] = comments_list
    json_output.append(post_dict)

json_path = "../dataset/vibe_coding_transformer_processed.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_output, f, indent=2)

print(f"Saved {len(json_output)} Posts and their filtered nested comments to {json_path}")

Saved 15633 Posts and their filtered nested comments to ../dataset/vibe_coding_transformer_processed.json
